In [36]:
# %%
# Cell 1: Global Configuration, Imports & Setup
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from dotenv import load_dotenv
from fredapi import Fred
import timesfm

# ==============================================================================
# GLOBAL CONFIGURATION - Change these variables to scale and adapt the script
# ==============================================================================

# Data Paths
DATA_FILE_PATH = "U20405-M - Sheet1.csv"

# Target Configuration
TARGET_KEYS = ['DVAPRC', 'DFSARC', 'DTRSRC', 'DRCARC', 'DGOERC', 'DPCERC']
PRIMARY_TARGET_SECTOR = 'DVAPRC'

# Macroeconomic Driver Configuration
FRED_DRIVER_SERIES = 'POILBREUSDM'       # e.g., POILBREUSDM for Brent Crude
DRIVER_NAME = 'Brend Crude'             # Generic column name for the exogenous driver
DRIVER_SHOCK_THRESHOLD = 100             # The value threshold to identify a 'shock'

# Model Configuration
MODEL_CONTEXT_LENGTH = 300
MODEL_HORIZON_LENGTH = 12

# Forecasting Weight Strategies (History vs. TimesFM Baseline)
MULTI_HISTORICAL_WEIGHT = 0.60           # Weight for multi-variate scenario blending
UNI_HISTORICAL_WEIGHT = 0.60             # Weight for direct univariate blending

# Visualization Configuration
PLOT_START_DATE = '2005-01-01'           # Determines how far back the charts render

# ==============================================================================

# Load environment variables
load_dotenv()

# Securely fetch the FRED API key
FRED_API_KEY = os.getenv('FRED_API_KEY')
if not FRED_API_KEY:
    raise ValueError("FRED_API_KEY not found. Please ensure your .env file is set up correctly.")

# Initialize FRED API client
fred = Fred(api_key=FRED_API_KEY)

# Configure clean, corporate plotting styles for business analysts
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (14, 7),
    'figure.autolayout': True,
    'axes.titlesize': 16,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'lines.linewidth': 2.5,
    'font.family': 'sans-serif',
    'legend.frameon': True,
    'legend.facecolor': 'white'
})

print("✅ Global configurations set, styling configured, and FRED API initialized successfully.")

✅ Global configurations set, styling configured, and FRED API initialized successfully.


In [37]:
# %%
# Cell 2: Model Instantiation
import torch
import timesfm

print("Initializing TimesFM 2.5 model...")

# Set matmul precision for hardware acceleration (Ampere+ GPUs)
torch.set_float32_matmul_precision("high")

# Instantiate the 2.5 model directly from HuggingFace
tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

# Compile the model with the correct forecast configuration using global variables
tfm.compile(
    timesfm.ForecastConfig(
        max_context=MODEL_CONTEXT_LENGTH,
        max_horizon=MODEL_HORIZON_LENGTH,
        normalize_inputs=True,              # Recommended to prevent scale instability
        use_continuous_quantile_head=True,  # Recommended for accurate quantiles
        return_backcast=True
    )
)

print(f"✅ TimesFM 2.5 instantiated and compiled successfully with a {MODEL_HORIZON_LENGTH}-month horizon.")

Initializing TimesFM 2.5 model...
✅ TimesFM 2.5 instantiated and compiled successfully with a 12-month horizon.


In [38]:
# %%
# Cell 3: Load Data & Convert to Long Format
print("Loading and reshaping dataset...")

try:
    # Read the wide-format CSV file using the global file path
    df_wide = pd.read_csv(DATA_FILE_PATH)
    print(f"✅ Successfully loaded '{DATA_FILE_PATH}' in wide format.")
    
    # Melt the dataframe from wide to long
    df_long = pd.melt(
        df_wide, 
        id_vars=['category', 'category_key'], # Keep identifiers intact
        var_name='Date_Raw',                  # The previous column headers (e.g., '1959M01')
        value_name='value'                    # The actual index values
    )
    
    # Convert the BEA 'YYYYMmm' format into proper pandas datetime objects
    # We replace 'M' with '-' to make it 'YYYY-mm', which pandas parses natively
    df_long['Date'] = pd.to_datetime(df_long['Date_Raw'].str.replace('M', '-'))
    
    # Drop the raw date string to keep it clean, sort chronologically, and drop missing values
    df_long = df_long.drop(columns=['Date_Raw']).sort_values(by=['category_key', 'Date']).dropna(subset=['value']).reset_index(drop=True)
    
    print(f"✅ Dataset successfully reshaped to long format.")
    print(f"New dataset shape: {df_long.shape[0]} rows, {df_long.shape[1]} columns.\n")
    
    # Display the first few rows to verify the transformation
    display(df_long.head())
    
except FileNotFoundError:
    print(f"❌ Error: Could not find '{DATA_FILE_PATH}'. Please ensure the file is in the same directory.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

Loading and reshaping dataset...
✅ Successfully loaded 'U20405-M - Sheet1.csv' in wide format.
✅ Dataset successfully reshaped to long format.
New dataset shape: 324414 rows, 4 columns.



,category,category_key,value,Date
0,Expenditures abroad by U.S. residents,DABDRC,"1,082",1959-01-01
1,Expenditures abroad by U.S. residents,DABDRC,"1,108",1959-02-01
2,Expenditures abroad by U.S. residents,DABDRC,"1,146",1959-03-01
3,Expenditures abroad by U.S. residents,DABDRC,"1,130",1959-04-01
4,Expenditures abroad by U.S. residents,DABDRC,"1,075",1959-05-01


In [39]:
# %%
# Cell 4: Clean & Align Multiple Variables
print("Cleaning and aligning datasets...")

# 1. Process target keys from the global config
df_targets = df_long[df_long['category_key'].isin(TARGET_KEYS)].copy()
df_targets['Date'] = pd.to_datetime(df_targets['Date'])

# Use pivot_table with aggfunc='first' to gracefully handle any duplicate rows in the raw CSV
df_targets = df_targets.pivot_table(
    index='Date', 
    columns='category_key', 
    values='value', 
    aggfunc='first'
)
df_targets = df_targets.resample('MS').first() # Force strict monthly frequency

# 2. Fetch Macro Driver dynamically from FRED
print(f"Fetching Macro Driver ({FRED_DRIVER_SERIES}) from FRED API...")
driver_series = fred.get_series(FRED_DRIVER_SERIES)
df_driver = pd.DataFrame({DRIVER_NAME: driver_series})
df_driver.index.name = 'Date'
df_driver.index = pd.to_datetime(df_driver.index)
df_driver = df_driver.resample('MS').mean()

# 3. Merge the datasets
df_clean = pd.merge(df_targets, df_driver, left_index=True, right_index=True, how='outer')

# 4. Handle gaps and trimming
df_clean[DRIVER_NAME] = df_clean[DRIVER_NAME].ffill()

# Trim to when we have data for the primary target index
first_valid_date = df_clean[PRIMARY_TARGET_SECTOR].first_valid_index()
df_clean = df_clean.loc[first_valid_date:]

# Ragged Edge preservation
df_clean = df_clean[df_clean[DRIVER_NAME].notna()]

print("✅ Data aligned successfully. Multi-variate structure established.")
display(df_clean.tail(3))

Cleaning and aligning datasets...
Fetching Macro Driver (POILBREUSDM) from FRED API...
✅ Data aligned successfully. Multi-variate structure established.


,DFSARC,DGOERC,DPCERC,DRCARC,DTRSRC,DVAPRC,Brend Crude
Date,,,,,,,
2026-01-01,"1,514,584","415,724","21,525,465","859,038","728,267","446,165",64.594091
2026-02-01,"1,522,911","422,429","21,665,077","850,537","744,523","451,196",69.409500
2026-03-01,"1,526,367","503,683","21,860,518","856,019","751,261","455,896",99.405000


In [40]:
# %%
# Cell 5: Extract High-Driver Historical Scenarios
import numpy as np
import pandas as pd

print(f"Identifying >{DRIVER_SHOCK_THRESHOLD} {DRIVER_NAME} periods & extracting scenario trajectories...")

# 1. BULLETPROOF NUMERIC CONVERSION
for col in df_clean.columns:
    df_clean[col] = df_clean[col].astype(str).str.replace(',', '', regex=False).str.strip()
    
df_clean = df_clean.apply(pd.to_numeric, errors='coerce')

# Identify covariates by excluding the primary target and the exogenous driver dynamically
intermediate_covariates = [col for col in df_clean.columns if col not in [PRIMARY_TARGET_SECTOR, DRIVER_NAME]]
print(f"✅ Dynamic Macro Drivers identified: {intermediate_covariates}")

# 2. Programmatically find the start of driver shocks based on global threshold
driver_above_threshold = df_clean[DRIVER_NAME] > DRIVER_SHOCK_THRESHOLD
start_of_spikes = driver_above_threshold & ~driver_above_threshold.shift(1, fill_value=False)
anchor_dates = df_clean.index[start_of_spikes]

# 3. Extract dynamic multipliers distinctively for EACH scenario 
scenario_shock_shapes = {}

print(f"Found {len(anchor_dates)} historical shocks crossing {DRIVER_SHOCK_THRESHOLD}:")
for anchor in anchor_dates:
    scenario_name = anchor.strftime('%Y-%b')
    print(f" - Initializing Scenario: {scenario_name}")
    
    # Dynamically scale the end date based on our horizon config
    end_date = anchor + pd.DateOffset(months=MODEL_HORIZON_LENGTH)
    scenario_shock_shapes[scenario_name] = {}
    
    for cov in intermediate_covariates:
        valid_cov_data = df_clean[cov].dropna()
        
        # SAFE LOOP: Ensure data exists for this specific historical timeframe
        if not valid_cov_data.empty and end_date <= valid_cov_data.index[-1]:
            slice_horizon = df_clean.loc[anchor:end_date, cov]
            
            # Check if we have enough data points (Horizon + 1 for the anchor baseline month)
            if len(slice_horizon) >= (MODEL_HORIZON_LENGTH + 1): 
                trajectory = (slice_horizon.values / slice_horizon.values[0])[1:]
                scenario_shock_shapes[scenario_name][cov] = trajectory
            else:
                scenario_shock_shapes[scenario_name][cov] = np.ones(MODEL_HORIZON_LENGTH) # Fallback
        else:
            scenario_shock_shapes[scenario_name][cov] = np.ones(MODEL_HORIZON_LENGTH) # Fallback

print("✅ Scenario extraction complete.")

Identifying >100 Brend Crude periods & extracting scenario trajectories...
✅ Dynamic Macro Drivers identified: ['DFSARC', 'DGOERC', 'DPCERC', 'DRCARC', 'DTRSRC']
Found 4 historical shocks crossing 100:
 - Initializing Scenario: 2008-Mar
 - Initializing Scenario: 2011-Feb
 - Initializing Scenario: 2012-Jul
 - Initializing Scenario: 2022-Mar
✅ Scenario extraction complete.


In [41]:
# %%
# Cell 6: Forecast Intermediaries per Scenario
print("Executing scenario-based forecasts for intermediate covariates...")

# Dynamically calculate the TimesFM weight based on the global historical weight
TIMESFM_WEIGHT = 1.0 - MULTI_HISTORICAL_WEIGHT
print(f"Weighting Strategy: {MULTI_HISTORICAL_WEIGHT*100}% Scenario History / {TIMESFM_WEIGHT*100}% TimesFM Baseline")

# Scale the date generation dynamically based on the global horizon length
future_dates = pd.date_range(start=df_clean.index[-1] + pd.DateOffset(months=1), periods=MODEL_HORIZON_LENGTH, freq='MS')

# 1. Pre-compute TimesFM Baselines (Only needs to run once)
timesfm_baselines = {}
baseline_values = {}

for cov in intermediate_covariates:
    history_array = df_clean[cov].dropna().values
    baseline_values[cov] = history_array[-1] # Save the jump-off point
    
    # Use global context length
    context_data = history_array[-MODEL_CONTEXT_LENGTH:]
    
    # Use global horizon length
    point_forecast, _ = tfm.forecast(horizon=MODEL_HORIZON_LENGTH, inputs=[context_data])
    timesfm_baselines[cov] = point_forecast[0][-MODEL_HORIZON_LENGTH:]

# 2. Blend the baseline with EACH scenario
scenario_cov_forecasts = {}

for scenario, shapes in scenario_shock_shapes.items():
    df_scenario = pd.DataFrame(index=future_dates)
    
    for cov in intermediate_covariates:
        pure_historical = baseline_values[cov] * shapes[cov]
        # Apply the global weighting strategy
        blended_forecast = (pure_historical * MULTI_HISTORICAL_WEIGHT) + (timesfm_baselines[cov] * TIMESFM_WEIGHT)
        df_scenario[cov] = blended_forecast
        
    scenario_cov_forecasts[scenario] = df_scenario

print(f"\n✅ {MODEL_HORIZON_LENGTH}-Month Covariate Forecasts generated for {len(scenario_cov_forecasts)} scenarios.")

Executing scenario-based forecasts for intermediate covariates...
Weighting Strategy: 60.0% Scenario History / 40.0% TimesFM Baseline

✅ 12-Month Covariate Forecasts generated for 4 scenarios.


In [42]:
# %%
# Cell 7: Execute Target XReg Forecast Iteratively
import numpy as np

print(f"Executing Multi-Scenario Target Forecasts for: {PRIMARY_TARGET_SECTOR}...")

# 1. Prepare Target Context Data
target_series = df_clean[PRIMARY_TARGET_SECTOR].dropna()
target_last_date = target_series.index[-1]

# Dynamically slice context using the global variable
target_context = target_series.values[-MODEL_CONTEXT_LENGTH:]
actual_context_len = len(target_context)
timesfm_target_input = [target_context.astype(np.float32)]

scenario_target_forecasts = {}

for scenario, df_cov_forecast in scenario_cov_forecasts.items():
    dynamic_cov_dict = {}
    
    # 2. Build dynamic covariates for THIS scenario
    for cov in intermediate_covariates:
        cov_history = df_clean.loc[:target_last_date, cov].ffill()
        cov_context = cov_history.values[-actual_context_len:]
        
        cov_future_values = []
        for d in future_dates:
            if d in df_clean.index and not pd.isna(df_clean.loc[d, cov]):
                cov_future_values.append(df_clean.loc[d, cov])
            elif d in df_cov_forecast.index:
                cov_future_values.append(df_cov_forecast.loc[d, cov])
            else:
                cov_future_values.append(cov_future_values[-1] if cov_future_values else cov_context[-1])
                
        cov_future = np.array(cov_future_values)
        full_cov_array = np.concatenate([cov_context, cov_future])
        dynamic_cov_dict[cov] = [full_cov_array.astype(np.float32)]

    # 3. Forecast Target using this scenario's exogenous features
    point_forecast, _ = tfm.forecast_with_covariates(
        inputs=timesfm_target_input,
        dynamic_numerical_covariates=dynamic_cov_dict,
        xreg_mode="xreg + timesfm" 
    )
    
    # 4. Store Results dynamically named
    df_target_forecast = pd.DataFrame(index=future_dates)
    df_target_forecast[f'{PRIMARY_TARGET_SECTOR}_Forecast'] = point_forecast[0]
    scenario_target_forecasts[scenario] = df_target_forecast

print(f"✅ Executed {len(scenario_target_forecasts)} distinct scenario forecasts.")

Executing Multi-Scenario Target Forecasts for: DVAPRC...
✅ Executed 4 distinct scenario forecasts.


In [43]:
# %%
# Cell 8: Interactive Scenario Dashboards
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

print("Generating Interactive Plotly Dashboards...")

# Dynamically slice historical data for the plot using the global start date
historical_slice = df_clean.loc[PLOT_START_DATE:].copy()
target_last_date = df_clean[PRIMARY_TARGET_SECTOR].dropna().index[-1]
last_target_val = df_clean[PRIMARY_TARGET_SECTOR].dropna().iloc[-1]

# Pull the exact mapping from df_long to dynamically name the target to avoid variable collisions
csv_names = dict(zip(df_long['category_key'], df_long['category']))
target_display_name = csv_names.get(PRIMARY_TARGET_SECTOR, PRIMARY_TARGET_SECTOR)

fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True, vertical_spacing=0.1,
    subplot_titles=(
        f"<b>{target_display_name} Forecast</b> (Scenario Sensitivity)",
        f"<b>Macroeconomic Drivers</b> (Scenario Forecasts)"
    )
)

# --- TOP CHART: HISTORICAL TARGET ---
fig.add_trace(go.Scatter(
    x=historical_slice.index, y=historical_slice[PRIMARY_TARGET_SECTOR],
    mode='lines', name=f'Historical {target_display_name}', 
    line=dict(color='black', width=3), legendgroup='History'
), row=1, col=1)

# Generate a color palette for the scenarios
scenario_names = list(scenario_target_forecasts.keys())
colors = px.colors.qualitative.Plotly 

# --- TOP CHART: FORECAST SCENARIOS ---
for idx, scenario in enumerate(scenario_names):
    color = colors[idx % len(colors)]
    plot_target = scenario_target_forecasts[scenario].copy()
    plot_target.loc[target_last_date] = last_target_val # Bridge gap
    plot_target = plot_target.sort_index()
    
    fig.add_trace(go.Scatter(
        x=plot_target.index, y=plot_target[f'{PRIMARY_TARGET_SECTOR}_Forecast'],
        mode='lines', name=f'{scenario} Scenario', 
        line=dict(color=color, width=2, dash='dash'), legendgroup=scenario
    ), row=1, col=1)

# --- BOTTOM CHART: MACRO DRIVER SCENARIOS ---
for cov in intermediate_covariates:
    display_name = csv_names.get(cov, cov)
    valid_cov_data = df_clean[cov].dropna()
    
    if valid_cov_data.empty:
        continue
        
    cov_last_date = valid_cov_data.index[-1]
    cov_last_val = valid_cov_data.iloc[-1]
    
    # Historical Covariates
    fig.add_trace(go.Scatter(
        x=historical_slice.index, y=historical_slice[cov],
        mode='lines', name=display_name, 
        line=dict(color='gray', width=1), showlegend=False, legendgroup='History'
    ), row=2, col=1)
    
    # Forecast Covariates per Scenario
    for idx, scenario in enumerate(scenario_names):
        color = colors[idx % len(colors)]
        if cov in scenario_cov_forecasts[scenario].columns:
            plot_cov = scenario_cov_forecasts[scenario][[cov]].copy()
            plot_cov.loc[cov_last_date] = cov_last_val # Bridge gap
            plot_cov = plot_cov.sort_index()
            
            fig.add_trace(go.Scatter(
                x=plot_cov.index, y=plot_cov[cov],
                mode='lines', name=f'{display_name} ({scenario})', 
                line=dict(color=color, width=1.5, dash='dot'), 
                showlegend=False, legendgroup=scenario # Groups legends so clicking one toggles all related traces
            ), row=2, col=1)

# Formatting
fig.add_vline(x=target_last_date, line_width=2, line_dash="dot", line_color="black", row='all', col='all')

fig.update_layout(
    height=900, template="plotly_white", hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()

Generating Interactive Plotly Dashboards...


In [44]:
# %%
# Cell 9: Direct Univariate Scenario Forecasting (Bypassing Intermediaries)
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

print(f"Executing Direct Univariate Scenario Forecasts for: {PRIMARY_TARGET_SECTOR}...")

# 1. Configuration
TIMESFM_WEIGHT_UNI = 1.0 - UNI_HISTORICAL_WEIGHT
print(f"Weighting Strategy: {UNI_HISTORICAL_WEIGHT*100}% Historical Scenario / {TIMESFM_WEIGHT_UNI*100}% TimesFM Baseline")

target_series = df_clean[PRIMARY_TARGET_SECTOR].dropna()
target_last_date = target_series.index[-1]
baseline_value = target_series.iloc[-1]

# 2. Extract Direct Historical Trajectories dynamically
driver_above_threshold = df_clean[DRIVER_NAME] > DRIVER_SHOCK_THRESHOLD
start_of_spikes = driver_above_threshold & ~driver_above_threshold.shift(1, fill_value=False)
anchor_dates = df_clean.index[start_of_spikes]

target_shock_shapes = {}

for anchor in anchor_dates:
    scenario_name = anchor.strftime('%Y-%b')
    # Dynamically scale the end date based on our horizon config
    end_date = anchor + pd.DateOffset(months=MODEL_HORIZON_LENGTH)
    
    if end_date <= target_series.index[-1]:
        slice_horizon = target_series.loc[anchor:end_date]
        # Check if we have enough data points (Horizon + 1 for the anchor baseline month)
        if len(slice_horizon) >= (MODEL_HORIZON_LENGTH + 1): 
            trajectory = (slice_horizon.values / slice_horizon.values[0])[1:]
            target_shock_shapes[scenario_name] = trajectory
            print(f" - Extracted '{scenario_name}' historical trajectory for {PRIMARY_TARGET_SECTOR}")

# 3. Generate TimesFM Baseline 
print("\nGenerating purely univariate TimesFM baseline...")
context_data = target_series.values[-MODEL_CONTEXT_LENGTH:]

point_forecast, _ = tfm.forecast(
    horizon=MODEL_HORIZON_LENGTH, 
    inputs=[context_data.astype(np.float32)]
)
timesfm_baseline = point_forecast[0][-MODEL_HORIZON_LENGTH:]

# 4. Blend Baseline with Historical Scenarios
future_dates = pd.date_range(start=target_last_date + pd.DateOffset(months=1), periods=MODEL_HORIZON_LENGTH, freq='MS')
direct_scenario_forecasts = {}

for scenario, shape in target_shock_shapes.items():
    pure_historical = baseline_value * shape
    blended_forecast = (pure_historical * UNI_HISTORICAL_WEIGHT) + (timesfm_baseline * TIMESFM_WEIGHT_UNI)
    
    df_scen = pd.DataFrame(index=future_dates)
    df_scen[f'{PRIMARY_TARGET_SECTOR}_Direct_Forecast'] = blended_forecast
    direct_scenario_forecasts[scenario] = df_scen

print(f"✅ Generated {len(direct_scenario_forecasts)} direct scenario forecasts.\n")


# 5. Plotting
csv_names = dict(zip(df_long['category_key'], df_long['category']))
target_display_name = csv_names.get(PRIMARY_TARGET_SECTOR, PRIMARY_TARGET_SECTOR)

historical_slice = target_series.loc[PLOT_START_DATE:]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=historical_slice.index, y=historical_slice.values,
    mode='lines', name=f'Historical {target_display_name}', 
    line=dict(color='black', width=3)
))

colors = px.colors.qualitative.Plotly 
for idx, scenario in enumerate(target_shock_shapes.keys()):
    color = colors[idx % len(colors)]
    
    plot_target = direct_scenario_forecasts[scenario].copy()
    plot_target.loc[target_last_date] = baseline_value
    plot_target = plot_target.sort_index()
    
    fig.add_trace(go.Scatter(
        x=plot_target.index, y=plot_target[f'{PRIMARY_TARGET_SECTOR}_Direct_Forecast'],
        mode='lines', name=f'{scenario} Scenario', 
        line=dict(color=color, width=2.5, dash='dash')
    ))

fig.add_vline(x=target_last_date, line_width=2, line_dash="dot", line_color="black")

fig.update_layout(
    title=f"<b>Direct Univariate Forecast for {target_display_name}</b><br><sup>Blended strictly with historical >{DRIVER_SHOCK_THRESHOLD} {DRIVER_NAME} period behaviors</sup>",
    height=600, template="plotly_white", hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()

Executing Direct Univariate Scenario Forecasts for: DVAPRC...
Weighting Strategy: 60.0% Historical Scenario / 40.0% TimesFM Baseline
 - Extracted '2008-Mar' historical trajectory for DVAPRC
 - Extracted '2011-Feb' historical trajectory for DVAPRC
 - Extracted '2012-Jul' historical trajectory for DVAPRC
 - Extracted '2022-Mar' historical trajectory for DVAPRC

Generating purely univariate TimesFM baseline...
✅ Generated 4 direct scenario forecasts.



In [ ]:
# %%
# Cell 10: Export Data in Horizontal Wide-by-Date Format (Multi & Univariate)
import pandas as pd

print(f"Transforming and exporting historical and scenario forecasts for {PRIMARY_TARGET_SECTOR}...")

# Dynamically map category names to prevent hardcoding across BOTH exports
csv_names = dict(zip(df_long['category_key'], df_long['category']))

# ==============================================================================
# PART 1: EXPORT MULTIVARIATE FORECASTS (Target + Intermediaries)
# ==============================================================================
print("\nProcessing Multivariate data...")
cols_to_export = [PRIMARY_TARGET_SECTOR] + intermediate_covariates

# Process Historical Baseline
df_hist_multi = df_clean[cols_to_export].copy().T
df_hist_multi['scenario'] = 'Historical Baseline'

# Process Each Forecast Scenario
scenario_dfs = []
for scenario_name in scenario_target_forecasts.keys():
    df_tgt = scenario_target_forecasts[scenario_name].rename(
        columns={f'{PRIMARY_TARGET_SECTOR}_Forecast': PRIMARY_TARGET_SECTOR}
    )
    df_cov = scenario_cov_forecasts[scenario_name]
    
    df_combined = pd.concat([df_tgt, df_cov], axis=1).T
    df_combined['scenario'] = scenario_name
    scenario_dfs.append(df_combined)

# Concatenate & Restructure Metadata
df_final_multi = pd.concat([df_hist_multi] + scenario_dfs)
df_final_multi.index.name = 'category_key'
df_final_multi = df_final_multi.reset_index()
df_final_multi.insert(0, 'category', df_final_multi['category_key'].map(csv_names).fillna(df_final_multi['category_key']))

# Clean Date Headers
date_cols_multi = [col for col in df_final_multi.columns if col not in ['category', 'category_key', 'scenario']]
formatted_date_cols_multi = [d.strftime('%Y-%m') if pd.notna(d) else str(d) for d in date_cols_multi]
df_final_multi = df_final_multi.rename(columns=dict(zip(date_cols_multi, formatted_date_cols_multi)))

# Sort neatly
final_col_order_multi = ['category', 'category_key', 'scenario'] + formatted_date_cols_multi
df_final_multi = df_final_multi[final_col_order_multi]
df_final_multi['sort_order'] = df_final_multi['scenario'].apply(lambda x: 0 if x == 'Historical Baseline' else 1)
df_final_multi = df_final_multi.sort_values(by=['category_key', 'sort_order', 'scenario']).drop(columns=['sort_order'])

# Export Multivariate CSV
export_multi_filename = f"{PRIMARY_TARGET_SECTOR}_Multivariate_Horizontal_Forecasts.csv"
df_final_multi.to_csv(export_multi_filename, index=False)
print(f"✅ Multivariate Data exported to '{export_multi_filename}' (Rows: {len(df_final_multi)})")


# ==============================================================================
# PART 2: EXPORT DIRECT UNIVARIATE FORECASTS (Target Only)
# ==============================================================================
print("Processing Direct Univariate data...")

# Process Historical Baseline
df_hist_uni = df_clean[[PRIMARY_TARGET_SECTOR]].copy().T
df_hist_uni['scenario'] = 'Historical Baseline'

# Process Each Direct Forecast Scenario
scenario_uni_dfs = []
for scenario_name in direct_scenario_forecasts.keys():
    df_tgt_uni = direct_scenario_forecasts[scenario_name].rename(
        columns={f'{PRIMARY_TARGET_SECTOR}_Direct_Forecast': PRIMARY_TARGET_SECTOR}
    )
    df_combined_uni = df_tgt_uni.T
    df_combined_uni['scenario'] = scenario_name
    scenario_uni_dfs.append(df_combined_uni)

# Concatenate & Restructure Metadata
df_final_uni = pd.concat([df_hist_uni] + scenario_uni_dfs)
df_final_uni.index.name = 'category_key'
df_final_uni = df_final_uni.reset_index()
df_final_uni.insert(0, 'category', df_final_uni['category_key'].map(csv_names).fillna(df_final_uni['category_key']))

# Clean Date Headers
date_cols_uni = [col for col in df_final_uni.columns if col not in ['category', 'category_key', 'scenario']]
formatted_date_cols_uni = [d.strftime('%Y-%m') if pd.notna(d) else str(d) for d in date_cols_uni]
df_final_uni = df_final_uni.rename(columns=dict(zip(date_cols_uni, formatted_date_cols_uni)))

# Sort neatly
final_col_order_uni = ['category', 'category_key', 'scenario'] + formatted_date_cols_uni
df_final_uni = df_final_uni[final_col_order_uni]
df_final_uni['sort_order'] = df_final_uni['scenario'].apply(lambda x: 0 if x == 'Historical Baseline' else 1)
df_final_uni = df_final_uni.sort_values(by=['category_key', 'sort_order', 'scenario']).drop(columns=['sort_order'])

# Export Univariate CSV
export_uni_filename = f"{PRIMARY_TARGET_SECTOR}_Univariate_Horizontal_Forecasts.csv"
df_final_uni.to_csv(export_uni_filename, index=False)
print(f"✅ Univariate Data exported to '{export_uni_filename}' (Rows: {len(df_final_uni)})")

# ==============================================================================
print("\n✅ All exports completed successfully.")

# ==============================================================================
# PART 3: VERIFY EXPORTS (Display Headers & First Rows)
# ==============================================================================
print(f"\n🔍 Verifying Multivariate Export: {export_multi_filename}")
# Reload from the saved CSV to ensure exactly what was written to disk is what we see
df_check_multi = pd.read_csv(export_multi_filename)
display(df_check_multi.head(3))

print(f"\n🔍 Verifying Univariate Export: {export_uni_filename}")
df_check_uni = pd.read_csv(export_uni_filename)
display(df_check_uni.head(3))

Transforming and exporting historical and scenario forecasts for DVAPRC...

Processing Multivariate data...
✅ Multivariate Data exported to 'DVAPRC_Multivariate_Horizontal_Forecasts.csv' (Rows: 30)
Processing Direct Univariate data...
✅ Univariate Data exported to 'DVAPRC_Univariate_Horizontal_Forecasts.csv' (Rows: 5)

✅ All exports completed successfully.

🔍 Verifying Multivariate Export: DVAPRC_Multivariate_Horizontal_Forecasts.csv


,category,category_key,scenario,1992-01,1992-02,1992-03,1992-04,1992-05,1992-06,1992-07,...,2026-06,2026-07,2026-08,2026-09,2026-10,2026-11,2026-12,2027-01,2027-02,2027-03
0,Food services and accommodations,DFSARC,Historical Baseline,284955.0,285438.0,284833.0,281367.0,282989.0,277513.0,280546.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Food services and accommodations,DFSARC,2008-Mar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.542997e+06,1.545055e+06,1.548926e+06,1.545541e+06,1.548112e+06,1.544602e+06,1.545605e+06,1.544350e+06,1.543561e+06,1.531546e+06
2,Food services and accommodations,DFSARC,2011-Feb,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.543078e+06,1.550787e+06,1.554148e+06,1.559243e+06,1.569904e+06,1.569956e+06,1.574610e+06,1.570219e+06,1.586206e+06,1.590063e+06



🔍 Verifying Univariate Export: DVAPRC_Univariate_Horizontal_Forecasts.csv


,category,category_key,scenario,1992-01,1992-02,1992-03,1992-04,1992-05,1992-06,1992-07,...,2026-06,2026-07,2026-08,2026-09,2026-10,2026-11,2026-12,2027-01,2027-02,2027-03
0,"Video, audio, photographic, and informat...",DVAPRC,Historical Baseline,59626.0,58998.0,58289.0,58321.0,58607.0,58857.0,59536.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Video, audio, photographic, and informat...",DVAPRC,2008-Mar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,461484.488065,463025.368587,456994.009353,453213.777997,450013.840638,447333.455773,444123.337468,451995.571746,451284.799273,440773.627906
2,"Video, audio, photographic, and informat...",DVAPRC,2011-Feb,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,459207.864864,461910.469603,462251.859923,463593.974714,462194.478718,471524.932046,469137.751796,464534.682161,465821.744355,471328.311821


: 